In [3]:
#------------------------------Import Libraries-----------------------
#------------------------------Set Color & Theme----------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
# Suppress warning messages for clean notebook execution
warnings.filterwarnings('ignore')

NAVY = '#1F3864'
AMBER = '#E8A33D'
PALETTE = [NAVY, AMBER, '#5B7DB1', '#F2C572', '#8FA6C9', '#C97F1E']

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['font.size'] = 10

In [4]:
#-------------------------Data Load From Excel----------------------------
#-------------------------659,318 Rows Import Across All 9 Tables---------

Customers = r"E:\Indian E-Commerce Sales & Customer Analytics\Excel Files\customers.csv"
df1 = pd.read_csv(Customers)
print(df1)

Products = r"E:\Indian E-Commerce Sales & Customer Analytics\Excel Files\products.csv"
df2 = pd.read_csv(Products)
print(df2)

Marketing_Campaigns = r"E:\Indian E-Commerce Sales & Customer Analytics\Excel Files\marketing_campaigns.csv"
df3 = pd.read_csv(Marketing_Campaigns)
print(df3)

Orders = r"E:\Indian E-Commerce Sales & Customer Analytics\Excel Files\orders.csv"
df4 = pd.read_csv(Orders)
print(df4)

Order_items = r"E:\Indian E-Commerce Sales & Customer Analytics\Excel Files\order_items.csv"
df5 = pd.read_csv(Order_items)
print(df5)

Payments = r"E:\Indian E-Commerce Sales & Customer Analytics\Excel Files\payments.csv"
df6 = pd.read_csv(Payments)
print(df6)

Returns = r"E:\Indian E-Commerce Sales & Customer Analytics\Excel Files\returns.csv"
df7 = pd.read_csv(Returns)
print(df7)

Shipments = r"E:\Indian E-Commerce Sales & Customer Analytics\Excel Files\shipments.csv"
df8 = pd.read_csv(Shipments)
print(df8)

Customer_reviews = r"E:\Indian E-Commerce Sales & Customer Analytics\Excel Files\customer_reviews.csv"
df9 = pd.read_csv(Customer_reviews)
print(df9)

      customer_id customer_signup_date  gender   age age_group  \
0      CUST100000           2023-01-09    Male  44.0     35-44   
1      CUST100001           2023-04-10    Male  20.0     18-24   
2      CUST100002           2025-05-09  Female  35.0     35-44   
3      CUST100003           2024-12-24  Female  34.0     25-34   
4      CUST100004           2023-08-15    Male  38.0     35-44   
...           ...                  ...     ...   ...       ...   
24995  CUST124995           2024-03-02    Male  33.0     25-34   
24996  CUST124996           2023-06-19  Female  52.0     45-54   
24997  CUST124997           2025-11-01    Male  18.0     18-24   
24998  CUST124998           2025-02-01    Male  35.0     35-44   
24999  CUST124999           2025-11-23  Female  51.0     45-54   

                state           city  pincode_prefix customer_segment  \
0      Madhya Pradesh         Indore             607          Regular   
1         West Bengal        Kolkata             245         

In [22]:
#--------------Data Overview & Quality Check---------------
tables_df = {
    'Customers' : df1,
    'Products' : df2,
    'Marketing_Campaigns' : df3,
    'Orders' : df4,
    'Order_items' : df5,
    'Payments' : df6,
    'Returns' : df7,
    'Shipments' : df8,
    'Customer_reviews' : df9
}

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77530 entries, 0 to 77529
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   review_id          77530 non-null  object
 1   order_id           77530 non-null  object
 2   customer_id        77530 non-null  object
 3   product_id         77530 non-null  object
 4   review_date        77530 non-null  object
 5   rating             77530 non-null  int64 
 6   review_sentiment   77297 non-null  object
 7   verified_purchase  77530 non-null  bool  
dtypes: bool(1), int64(1), object(6)
memory usage: 4.2+ MB


In [24]:
df.describe()

,rating
count,77530.000000
mean,3.691939
std,0.904587
min,1.000000
25%,3.000000
50%,4.000000
75%,4.000000
max,5.000000


In [25]:
df.head()

,review_id,order_id,customer_id,product_id,review_date,rating,review_sentiment,verified_purchase
0,REV100000,ORD1000002,CUST120793,PROD10470,2024-12-30,4,Positive,True
1,REV100001,ORD1000005,CUST123408,PROD10085,2025-02-04,4,Positive,True
2,REV100002,ORD1000005,CUST123408,PROD10233,2025-01-27,5,Positive,True
3,REV100003,ORD1000006,CUST117355,PROD10176,2025-06-11,4,Positive,True
4,REV100004,ORD1000006,CUST117355,PROD10210,2025-06-06,3,Neutral,True


In [8]:
#------------------1. Shape, missing values, duplicates summary--------------
overview = pd.DataFrame({
    'rows': {k: len(v) for k, v in tables_df.items()},
    'columns': {k: v.shape[1] for k, v in tables_df.items()},
    'missing_values': {k: v.isnull().sum().sum() for k, v in tables_df.items()},
    'duplicate_rows': {k: v.duplicated().sum() for k, v in tables_df.items()}
})
print(overview.sort_values('rows', ascending=False))
print(f"\nTotal rows across all files: {sum(len(t) for t in tables_df.values()):,}")

                       rows  columns  missing_values  duplicate_rows
Order_items          254331        9               0               0
Payments             100000        8               0               0
Orders               100000       16          116532               0
Shipments             89681       11            4642               0
Customer_reviews      77530        8             233               0
Customers             25000       18            2675               0
Returns               12075        8               0               0
Products                550       14               4               0
Marketing_Campaigns     151       14               0               0

Total rows across all files: 659,318


In [18]:
# 2. Data types per table
for name, df in tables_df.items():
    print(f"\n--- {name} dtypes ---")
    print(df.dtypes)

# 3. Missing values breakdown (only columns with nulls)
print("=== MISSING VALUES BY COLUMN ===")
for name, df in tables_df.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f"\n--- {name} ---")
        print(nulls)
        print(f"% missing: {(nulls / len(df) * 100).round(2).to_dict()}")


--- Customers dtypes ---
customer_id                  object
customer_signup_date         object
gender                       object
age                         float64
age_group                    object
state                        object
city                         object
pincode_prefix                int64
customer_segment             object
preferred_device             object
preferred_payment_method     object
acquisition_channel          object
total_orders                  int64
total_spend                 float64
last_order_date              object
average_order_value         float64
customer_status              object
loyalty_tier                 object
dtype: object

--- Products dtypes ---
product_id               object
product_name             object
category                 object
subcategory              object
brand                    object
price                   float64
cost_price              float64
discount_range           object
rating_average          float64

In [30]:
valid_orders = Orders[Orders['order_status'] != 'Failed'].copy()
valid_orders['year_month'] = valid_orders['order_date'].dt.to_period('M').astype(str)

monthly = valid_orders.groupby('year_month').agg(
    revenue=('final_amount', 'sum'),
    orders=('order_id', 'count')
).reset_index()

fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.bar(monthly['year_month'], monthly['revenue'], color=NAVY, alpha=0.85, label='Revenue')
ax1.set_ylabel('Revenue (₹)', color=NAVY)
ax1.tick_params(axis='x', rotation=90)
ax2 = ax1.twinx()
ax2.plot(monthly['year_month'], monthly['orders'], color=AMBER, marker='o', linewidth=2, label='Orders')
ax2.set_ylabel('Order Count', color=AMBER)
plt.title('Monthly Revenue & Order Volume', fontsize=13, color=NAVY, fontweight='bold')
fig.tight_layout()
plt.show()

TypeError: string indices must be integers, not 'str'